# Synthetic datasets

## Exercise 4 (Regresssion)

**TODO**: maybe even simulate names/employee-ids?

In [4]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300

departments = np.random.choice(["Sales", "IT", "HR"], size=n, p=[0.4, 0.4, 0.2])

# Core features
experience = np.random.randint(0, 21, n)
training_hours = np.random.normal(40, 10, n).clip(5, 80)
team_size = np.random.randint(3, 15, n)
remote_share = np.random.beta(2.2, 2.0, n)  # slightly balanced around moderate remote work

# Department-specific manager quality distributions
# IT has more high-quality managers on average
manager_quality = np.zeros(n, dtype=int)
for i, d in enumerate(departments):
    if d == "Sales":
        manager_quality[i] = np.random.choice([1, 2, 3, 4, 5], p=[0.12, 0.23, 0.30, 0.22, 0.13])
    elif d == "IT":
        manager_quality[i] = np.random.choice([1, 2, 3, 4, 5], p=[0.04, 0.08, 0.18, 0.36, 0.34])
    else:  # HR
        manager_quality[i] = np.random.choice([1, 2, 3, 4, 5], p=[0.18, 0.28, 0.27, 0.17, 0.10])

salary = 30000 + experience * 2000 + np.random.normal(0, 5000, n)

dept_effect = {"Sales": 10, "IT": 5, "HR": 0}
dept_numeric = np.array([dept_effect[d] for d in departments])

# Centered terms for interactions
training_c = training_hours - training_hours.mean()
remote_c = remote_share - remote_share.mean()

# Use department-specific centering for manager quality so the interaction
# is more visible within each department subset
manager_c_within = np.zeros(n)
for d in ["Sales", "IT", "HR"]:
    idx = departments == d
    manager_c_within[idx] = manager_quality[idx] - manager_quality[idx].mean()

# Hidden interaction: training works better with good managers
training_manager_interaction = 0.16 * training_c * manager_c_within

# Main effect of remote is small overall
remote_main = -0.8 * remote_c

# Department-specific direct remote effects:
# Sales/HR somewhat negative, IT slightly positive
remote_dept_main = {"Sales": -4.0, "IT": 2.5, "HR": -5.0}
remote_dept_component = np.array(
    [remote_dept_main[d] for d in departments]
) * remote_c

# Make the interaction more evident, but keep total remote slopes moderate
# Interpretation:
# - In IT, good managers make remote work work well
# - In Sales/HR, weak managers make remote work more problematic
remote_interaction_strength = {"Sales": 9.0, "IT": 12.0, "HR": 10.0}
remote_manager_interaction = np.array(
    [remote_interaction_strength[d] for d in departments]
) * remote_c * manager_c_within

# Stronger direct manager-quality effect in IT so that it aligns with the
# more favorable remote-work context there
manager_main_strength = {"Sales": 4.8, "IT": 6.8, "HR": 4.0}
manager_component = np.array(
    [manager_main_strength[d] for d in departments]
) * manager_quality

performance = (
    50
    + 1.8 * experience
    + 0.9 * training_hours
    + 0.0004 * salary
    + 1.4 * team_size
    + manager_component
    + remote_main
    + remote_dept_component
    + training_manager_interaction
    + remote_manager_interaction
    + dept_numeric
    + np.random.normal(0, 9, n)
)

df = pd.DataFrame({
    "performance": performance,
    "experience": experience,
    "training_hours": training_hours,
    "salary": salary,
    "remote_share": remote_share,
    "team_size": team_size,
    "manager_quality": manager_quality,
    "department": departments
})

df.to_csv("employee_performance_data.csv", index=False)
print(df.head())
print(df.groupby("department")[["manager_quality", "remote_share", "performance"]].mean())

   performance  experience  training_hours        salary  remote_share  \
0   159.151624          12       32.603812  60215.917186      0.879245   
1   168.470117          19       33.507527  63505.934060      0.182947   
2   164.861349          14       31.514793  54472.871760      0.552633   
3   159.413368           2       30.637550  34311.578227      0.513527   
4   145.708783           7       43.331733  49572.581421      0.663745   

   team_size  manager_quality department  
0         11                3      Sales  
1          4                5         HR  
2          7                3         IT  
3         14                5         IT  
4          5                2      Sales  
            manager_quality  remote_share  performance
department                                            
HR                 2.650794      0.498049   148.669269
IT                 3.953704      0.515782   170.112483
Sales              3.162791      0.585727   162.632426
